# Ingestion: slide chunks to Qdrant (hybrid search)

Importing the libraries and putting the model names and the data path in one place, so everything stays fixed and reproducible

In [ ]:
from pathlib import Path
import json

from fastembed import SparseTextEmbedding
from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient, models

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "parsed_clean_DELETED"

QDRANT_URL = "http://localhost:6333"
COLLECTION = "lecture_chunks"

DENSE_MODEL = "BAAI/bge-m3"
SPARSE_MODEL = "Qdrant/bm25"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"

print("Project root:", PROJECT_ROOT)
print("Data dir exists:", DATA_DIR.exists())

Opening the client and listing the existing collections to check that Qdrant is reachable before indexing anything

In [ ]:
client = QdrantClient(url=QDRANT_URL)
print("Existing Collections:", [c.name for c in client.get_collections().collections])

## Load and inspect a single chunk

Loading one chunk file and printing a single chunk to see the payload schema (title, page_content, metadata) that retrieval depends on later

In [ ]:
chunks_file = next(DATA_DIR.rglob("*chunks*.json"))
with chunks_file.open("r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Number of Chunks in this file: {len(chunks)}")
print(json.dumps(chunks[0], indent=4, ensure_ascii=False))

## Build the embedding text

build_embedding_text glues title, content and context into the string we actually embed, the same fields for every chunk

In [ ]:
def build_embedding_text(chunk: dict) -> str:
    parts = []
    if chunk.get("title"):
        parts.append(chunk["title"])
    if chunk.get("page_content"):
        parts.append(chunk["page_content"])
    if chunk.get("context"):
        parts.append(chunk["context"])
    return "\n\n".join(parts)

sample_text = build_embedding_text(chunks[0])
print(f"Length of embedding_text: {len(sample_text)} characters")
print("\nDISPLAYING FIRST 600 CHARACTERS OF THE EMBEDDING TEXT:\n ")
print(sample_text[:600], "...")

## Dense embedding (BGE-M3)

BGE-M3 runs through sentence-transformers on the GPU (~2.3 GB, gets cached in ~/.cache/huggingface). I set normalize_embeddings=True so we get L2 normalised vectors, which is what the cosine distance in Qdrant needs

Embedding the sample text and checking that we really get a 1024 dim normalised vector

In [ ]:
dense_embedder = SentenceTransformer(DENSE_MODEL, device="cuda")

dense_vec = dense_embedder.encode(sample_text, normalize_embeddings=True)

print("Shape:", dense_vec.shape)
print("Dtype:", dense_vec.dtype)
print("First 8 values:", dense_vec[:8])

BGE-M3 returns normalised vectors

These 1024 numbers are the semantic representation of the slide

## Sparse embedding (BM25)

BM25 gives every token a score (TF with document length normalisation, times IDF). The IDF has to come from the corpus. fastembed only ships the tokenizer config, the actual IDF is computed inside Qdrant at query time via Modifier.IDF

Embedding the same text with BM25 to get a token/score pair per term, the lexical counterpart to the dense vector

In [ ]:
sparse_embedder = SparseTextEmbedding(model_name=SPARSE_MODEL, language="german")

sparse_vec = list(sparse_embedder.embed([sample_text]))[0]

print("Type:", type(sparse_vec).__name__)
print("Number of Non-Zero Tokens:", len(sparse_vec.indices))
print("First 10 Token IDs:    ", sparse_vec.indices[:10])
print("First 10 BM25 Values:   ", sparse_vec.values[:10])

## Create the Qdrant collection

- dense: 1024 floats, cosine distance
- sparse: BM25 with Modifier.IDF, so Qdrant weights the terms with corpus wide IDF at search time

Creating the collection with the dense and the sparse vector defined above

In [ ]:
if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)
    print("Deleted existing collection")

client.create_collection(
    collection_name=COLLECTION,
    vectors_config={
        "dense": models.VectorParams(size=1024, distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(modifier=models.Modifier.IDF),
    },
)

info = client.get_collection(COLLECTION)
print("Status:", info.status)
print("Vectors config:", info.config.params.vectors)
print("Sparse config: ", info.config.params.sparse_vectors)

## Upsert this one chunk

First only a single point, just to check that the data structure is right

Upserting one point to validate the structure (deterministic UUID, both vectors, payload) before doing the whole corpus

In [ ]:
import uuid

point_id = str(uuid.uuid5(uuid.NAMESPACE_URL, chunks[0]["id"]))

point = models.PointStruct(
    id=point_id,
    vector={
        "dense": dense_vec.tolist(),
        "sparse": models.SparseVector(
            indices=sparse_vec.indices.tolist(),
            values=sparse_vec.values.tolist(),
        ),
    },
    payload={
        "chunk_id": chunks[0]["id"],
        "modul": chunks[0]["modul"],
        "lecture": chunks[0]["lecture"],
        "title": chunks[0]["title"],
        "page_numbers": chunks[0]["page_numbers"],
        "page_content": chunks[0]["page_content"],
        "context": chunks[0]["context"],
        "page_reference_path": chunks[0]["page_reference_path"]
    },
)

client.upsert(collection_name=COLLECTION, points=[point])
print("Anzahl Points jetzt:", client.count(COLLECTION).count)

## Test query against this one point

Dense search only for now

A dense only test query against the single point to confirm that encoding, storage and scoring work

In [ ]:
query = "Von wem ist das Modul?"

q_dense = dense_embedder.encode(query, normalize_embeddings=True)

hits = client.query_points(
    collection_name=COLLECTION,
    query=q_dense.tolist(),
    using="dense",
    limit=3,
    with_payload=True,
).points

for h in hits:
    print(f"score={h.score:.4f} | {h.payload['title']} (p.{h.payload['page_numbers']})")

print(hits[0].payload["page_content"][:100], "...")

## Now all chunks in a loop

The mechanism is clear, so now the whole thing for real. Embedding in batches is faster than one chunk at a time

Loading every chunk file across all lectures, building the embedding texts and printing a length stat to see how much the chunk size varies

In [ ]:
all_chunks = []
for chunk in DATA_DIR.rglob("*.json"):
    with chunk.open("r", encoding="utf-8") as f:
        all_chunks.extend(json.load(f))

print(f"Overall Chunks: {len(all_chunks)}")

texts = [build_embedding_text(c) for c in all_chunks]
print(f"Length Statistics (Characters): min={min(len(t) for t in texts)}, max={max(len(t) for t in texts)}")

Batch embedding the whole corpus with both encoders, which is much faster than per chunk, and printing the average sparse token count

In [ ]:
dense_vecs = list(dense_embedder.encode(texts, batch_size=8, normalize_embeddings=True))
sparse_vecs = list(sparse_embedder.embed(texts, batch_size=8))

print("Dense:", len(dense_vecs), "× shape", dense_vecs[0].shape)
print("Sparse:", len(sparse_vecs), "×  nonzero =",
      round(sum(len(s.indices) for s in sparse_vecs) / len(sparse_vecs), 1))

Recreating the collection from scratch and upserting all points, so we end up with a clean, fully indexed corpus

In [ ]:
if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)
client.create_collection(
    collection_name=COLLECTION,
    vectors_config={"dense": models.VectorParams(size=1024, distance=models.Distance.COSINE)},
    sparse_vectors_config={"sparse": models.SparseVectorParams(modifier=models.Modifier.IDF)},
)

points = []
for chunk, dv, sv in zip(all_chunks, dense_vecs, sparse_vecs):
    points.append(models.PointStruct(
        id=str(uuid.uuid5(uuid.NAMESPACE_URL, chunk["id"])),
        vector={
            "dense": dv.tolist(),
            "sparse": models.SparseVector(indices=sv.indices.tolist(), values=sv.values.tolist()),
        },
        payload={
            "chunk_id": chunk["id"],
            "modul": chunk["modul"],
            "lecture": chunk["lecture"],
            "page_reference_path": chunk["page_reference_path"],
            "title": chunk["title"],
            "page_numbers": chunk["page_numbers"],
            "page_content": chunk["page_content"],
            "context": chunk["context"],
        },
    ))

client.upsert(collection_name=COLLECTION, points=points)
print("Indexed:", client.count(COLLECTION).count)

## Hybrid search with RRF

The Qdrant Query API runs two separate prefetches (dense and sparse), each returns its own ranking. FusionQuery with Fusion.RRF combines them via Reciprocal Rank Fusion, which prefers documents that sit high in both lists

hybrid_search runs the two prefetches and fuses them with RRF

In [ ]:
def hybrid_search(query: str, limit: int = 5):
    q_dense = dense_embedder.encode(query, normalize_embeddings=True)
    q_sparse = list(sparse_embedder.query_embed(query))[0]

    return client.query_points(
        collection_name=COLLECTION,
        prefetch=[
            models.Prefetch(
                query=q_dense.tolist(), 
                    using="dense", 
                    limit=20,
            ),
            models.Prefetch(
                query=models.SparseVector(
                    indices=q_sparse.indices.tolist(),
                    values=q_sparse.values.tolist(),
                ),
                using="sparse",
                limit=20,
            ),
        ],
        
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=limit,
        with_payload=True,
    ).points

for q in [
    "Wie funktioniert eine Faltung und warum ist das relevant?",
]:
    print(f"\n=== {q} ===")
    for i, h in enumerate(hybrid_search(q), 1):
        print(f"  [{i}] {h.score:.4f} | {h.payload['lecture']} p.{h.payload['page_numbers']} | {h.payload['title']}")

## Ablation: hybrid vs dense only vs sparse only

Running the same query through dense only, sparse only and hybrid to compare, this is what motivates choosing hybrid

In [ ]:
def dense_only(query, limit=5):
    q = dense_embedder.encode(query, normalize_embeddings=True)
    return client.query_points(
        collection_name=COLLECTION, query=q.tolist(), using="dense",
        limit=limit, with_payload=True,
    ).points

def sparse_only(query, limit=5):
    q = list(sparse_embedder.query_embed(query))[0]
    return client.query_points(
        collection_name=COLLECTION,
        query=models.SparseVector(indices=q.indices.tolist(), values=q.values.tolist()),
        using="sparse", limit=limit, with_payload=True,
    ).points

q = "Wie funktioniert eine Faltung und warum ist das relevant?"
print(f"Query: {q}\n")

for name, fn in [("Dense", dense_only), ("Sparse (BM25)", sparse_only), ("Hybrid (RRF)", hybrid_search)]:
    print(f"--- {name} ---")
    for i, h in enumerate(fn(q), 1):
        print(f"  [{i}] {h.score:.4f} | {h.payload['lecture']} p.{h.payload['page_numbers']} | {h.payload['title']}")
    print()